---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\Doina\Desktop\ADC\AN 2\8. INGINERIE AI\Proiect\echochamber-app
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [3]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[CălinGeorgescu-CanalulOficial] Respect stima domnule presedinte...Romania inca doarme...Dumnezeu sa va dea sana
[CălinGeorgescu-CanalulOficial] Pacea de la București este morandumul inițiat de către Diana Șosoaca. Văd că vă 
[@CălinGeorgescu-CanalulOficial] Ajunge...domnule Georgescu deschideți o asociație dacă nu puteți un partid acum.


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [4]:
# modifica dupa preferinte

AXA_1 = "media_distrust"
AXA_2 = "elite_blame"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [5]:
AXA_1_DEFINITION = """
media_distrust măsoară neîncrederea în televiziuni,
presă, jurnaliști sau media mainstream.

0 = absent
1 = slab
2 = moderat
3 = puternic
"""
AXA_2_DEFINITION = """
elite_blame măsoară cât de mult comentariul
învinovățește politicienii, elitele sau instituțiile
pentru problemele societății.

0 = absent
1 = indirect
2 = clar
3 = foarte puternic
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [6]:
MINI_PROMPT = f"""
Ești un specialist in alaiza discursului politic de pe You Tube.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2 / 3
{AXA_2} = 0 / 1 / 2 / 3
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = slab, 2 = moderat, 3 = puternic
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un specialist in alaiza discursului politic de pe You Tube.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. media_distrust
2. elite_blame
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
media_distrust = 0 / 1 / 2 / 3
elite_blame = 0 / 1 / 2 / 3
DEFINIȚII:

media_distrust măsoară neîncrederea în televiziuni,
presă, jurnaliști sau media mainstream.

0 = absent
1 = slab
2 = moderat
3 = puternic


elite_blame măsoară cât de mult comentariul
învinovățește politicienii, elitele sau instituțiile
pentru problemele societății.

0 = absent
1 = indirect
2 = clar
3 = foarte puternic

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, 

## Pasul 4 — Alege 10 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [9]:
TESTS = corpus.sample(10, random_state=42)
TESTS[["id", "source_channel", "video_title", "text"]].head(10)

,id,source_channel,video_title,text
145,yt_BfvZ8QcVBKc_Ugyog0iMEqAX5zIQb4R4AaABAg,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Da, si eu cred ca serviciile ucrainene au fost..."
334,yt_qkGhsJFft00_UgzJHKCoGkwtTJ3uOBh4AaABAg,digi24hd56,În fața ta cu Emil Hurezeanu: „Ar fi un coșmar...,Ce mă supără pe mine oamenii ăștia care sunt a...
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
369,yt_vkP6FdP9iX0_Ugwj3HojTt6JikJVtjd4AaABAg,turcescu111,Orientul Mijlociu în flăcări,Nicușor merge pe lângă covor pentru că nu are ...
416,yt_Sj4fQKlMOro_UgyNWIwNDtUeTTCrLkl4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Trebuie susținută aceasta femeie!!!!! Acesta a...
73,yt_q6wy7cs5RuU_UgxtjWylQ8u3U0mA5u14AaABAg,RecorderRomania,Judecătoarea Raluca Moroșanu: „Dacă nu se schi...,Cat adevar cate dovezii ne prezentati si incon...
132,yt_jxIZUCUPnAw_UgxdtlDlSW5rAN7yAsN4AaABAg,turcescu111,"A fost rupere în audiență, pe asta n-au cum s-...","ROMANII VOR UN OM ONEST, CARE IUBESTE ROMANIA...."
137,yt_6hgc90sLVFw_Ugzw4PlIjKmVEAI97qt4AaABAg,spotmediaro,Oamenii apropiați de Moscova și-au păstrat inf...,"Ce bine spunea Rudyard Kipling, ... parintele ..."
30,yt_2lB4tTcjHzw_Ugz0V4O1fpH_HwAB2-F4AaABAg,euronewsro,Viktor Orban acuză UE și Ucraina că încearcă s...,Ii e frica lui Orban ca perioada lui de domnie...
72,yt_yEuctxNb4O0_UgxPNjxaWPqCBL5WUAB4AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,Bun! S-a desfășurat aceasta întâlnire. S-a lua...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [14]:
USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: deepseek-chat


In [15]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [17]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=500)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt infiltrate de serviciile ucrainene

OUTPUT MODEL:
{
  "target": "serviciile ucrainene",
  "stance": "anti",
  "tone": "acuzator",
  "media_distrust": 0,
  "elite_blame": 0
}
COMENTARIU:
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama dracului știu ei treburile astea. Noi bănuim ca primesc arme din Rusia bolshevica și China comunistă. Poate și Brazilia dar iarăși e o bănuială.

OUTPUT MODEL:
{
  "target": "mass media",
  "stance": "anti",
  "tone": "acuzator",
  "media_distrust": 3,
  "elite_blame": 0
}
COMENTARIU:
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️

OUTPUT MODEL:
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "media_distrust": 0,
  "elite_blame": 0
}
COMENTARIU:
Nicușor merge pe lângă covor pentru că nu are ,, stabi

In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

Am ales axele media_distrust si elite_blame deoarece apar frecvent in discursul politic online si pot evidentia neincrederea in mass-media si tendinta de a acuza elitele sau institutiile.

Le-am ales deoarece sunt usor de identificat in comentarii si produc diferente clare intre exemple.

Modelul a returnat de cele mai multe ori JSON corect si bine structurat.

Cea mai mare problema a fost ca uneori raspunsul era incomplet sau outputul era taiat.

As modifica promptul pentru a cere explicit un singur obiect JSON valid, fara explicatii suplimentare si fara simboluri precum "...".